In [2]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

# Read from LAMMPS data file
ld = LammpsData.from_file('bulk_structure/mg21zn25_relaxed.data', atom_style='atomic')
adaptor = AseAtomsAdaptor()
alloy = adaptor.get_atoms(ld.structure)

print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")
print(f"Cell: {np.round(alloy.cell.lengths(), 3)} A")
print(f"Angles: {np.round(alloy.cell.angles(), 2)} deg")

Bulk: 276 atoms, Mg126Zn150
Cell: [26.121 26.121  8.788] A
Angles: [ 90.  90. 120.] deg


In [3]:
slab = surface(alloy, (0, 0, 1), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need {25/21:.4f})")
print(f"Stoichiometric: {'YES' if zn*21 == mg*25 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 552, Mg: 252, Zn: 300
Thickness: 17.6 A
Ratio Zn/Mg: 1.1905 (need 1.1905)
Stoichiometric: YES
Symmetric: False


In [4]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
print(f"Min gap: {gaps.min():.4f} A, Median gap: {np.median(gaps):.4f} A")
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Using tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Min gap: 0.0000 A, Median gap: 0.0000 A
Using tolerance: 0.068 A

Total atomic layers: 73

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0        Zn4     0.002  |       72        Zn3     0.000 NO <---
      1        Mg3     0.445  |       71        Mg3     0.442    YES
      2        Mg3     0.564  |       70        Mg3     0.561    YES
      3    Mg9Zn18     0.732  |       69    Mg9Zn18     0.729    YES
      4        Mg3     0.900  |       68        Mg3     0.898    YES
      5        Mg3     1.020  |       67        Mg3     1.017    YES
      6        Zn7     1.465  |       66        Zn7     1.462    YES
      7        Mg3     1.910  |       65        Mg3     1.907    YES
      8        Mg3     2.029  |       64        Mg3     2.026    YES
      9    Mg9Zn18     2.197  |       63    Mg9Zn18     2.194    YES
     10        Mg3     2.365  |       62        Mg3     2.362    YES

In [5]:
top_mask = z > z_max - 0.1
keep = np.where(~top_mask)[0]
trimmed = slab[keep]

ps_trim = AseAtomsAdaptor().get_structure(trimmed)
slab_trim = Slab(ps_trim.lattice, ps_trim.species, ps_trim.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps_trim, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

sym_t = np.array(trimmed.get_chemical_symbols())
print(f"After removing top Zn3:")
print(f"Atoms: {len(trimmed)}, Mg: {sum(sym_t=='Mg')}, Zn: {sum(sym_t=='Zn')}")
print(f"Symmetric: {slab_trim.is_symmetric()}")

if slab_trim.is_symmetric():
    sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
    print(f"Space group: {sga.get_space_group_symbol()}")
    for op in sga.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break

if not slab_trim.is_symmetric():
    # Try removing bottom Zn4 instead
    bot_mask = z < z_min + 0.1
    keep_bot = np.where(~bot_mask)[0]
    trimmed_bot = slab[keep_bot]
    
    ps_trim_bot = AseAtomsAdaptor().get_structure(trimmed_bot)
    slab_trim_bot = Slab(ps_trim_bot.lattice, ps_trim_bot.species, ps_trim_bot.frac_coords,
        miller_index=(0,0,1), oriented_unit_cell=ps_trim_bot, shift=0,
        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
    
    sym_tb = np.array(trimmed_bot.get_chemical_symbols())
    print(f"\nAfter removing bottom Zn4:")
    print(f"Atoms: {len(trimmed_bot)}, Mg: {sum(sym_tb=='Mg')}, Zn: {sum(sym_tb=='Zn')}")
    print(f"Symmetric: {slab_trim_bot.is_symmetric()}")
    
    if slab_trim_bot.is_symmetric():
        sga = SpacegroupAnalyzer(ps_trim_bot, symprec=0.1)
        print(f"Space group: {sga.get_space_group_symbol()}")
        for op in sga.get_symmetry_operations():
            if op.rotation_matrix[2][2] < 0:
                inv_translation = op.translation_vector
                print(f"Inversion: f -> {inv_translation} - f")
                break

    # Try removing both
    both_mask = top_mask | bot_mask
    keep_both = np.where(~both_mask)[0]
    trimmed_both = slab[keep_both]
    
    ps_trim_both = AseAtomsAdaptor().get_structure(trimmed_both)
    slab_trim_both = Slab(ps_trim_both.lattice, ps_trim_both.species, ps_trim_both.frac_coords,
        miller_index=(0,0,1), oriented_unit_cell=ps_trim_both, shift=0,
        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
    
    sym_tb2 = np.array(trimmed_both.get_chemical_symbols())
    print(f"\nAfter removing both top Zn3 AND bottom Zn4:")
    print(f"Atoms: {len(trimmed_both)}, Mg: {sum(sym_tb2=='Mg')}, Zn: {sum(sym_tb2=='Zn')}")
    print(f"Symmetric: {slab_trim_both.is_symmetric()}")
    
    if slab_trim_both.is_symmetric():
        sga = SpacegroupAnalyzer(ps_trim_both, symprec=0.1)
        print(f"Space group: {sga.get_space_group_symbol()}")
        for op in sga.get_symmetry_operations():
            if op.rotation_matrix[2][2] < 0:
                inv_translation = op.translation_vector
                print(f"Inversion: f -> {inv_translation} - f")
                break

After removing top Zn3:
Atoms: 549, Mg: 252, Zn: 297
Symmetric: False

After removing bottom Zn4:
Atoms: 548, Mg: 252, Zn: 296
Symmetric: False

After removing both top Zn3 AND bottom Zn4:
Atoms: 545, Mg: 252, Zn: 293
Symmetric: True
Space group: P-3
Inversion: f -> [0. 0. 0.] - f


In [6]:
inv_translation = np.array([0., 0., 0.])

# Get removed atoms
top_removed = [i for i in range(len(slab)) if z[i] > z_max - 0.1]
bot_removed = [i for i in range(len(slab)) if z[i] < z_min + 0.1]
all_removed = bot_removed + top_removed

print(f"Bottom removed ({len(bot_removed)}):")
for j in bot_removed:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

print(f"\nTop removed ({len(top_removed)}):")
for j in top_removed:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

print(f"\nTotal removed: {len(all_removed)} Zn")

# Check inversion partners
ps_orig = AseAtomsAdaptor().get_structure(slab)
keep_both = np.where(~(top_mask | bot_mask))[0]
keep_set = set(keep_both.tolist())

print(f"\nInversion partners:")
for j in all_removed:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    dists = np.linalg.norm(slab.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    closest = min(keep_set, key=lambda k: dists[k])
    
    if min_d < 0.5:
        print(f"  idx={j} -> paired with idx={closest} dist={min_d:.3f}")
    else:
        print(f"  idx={j} -> UNPAIRED, inv at ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

Bottom removed (4):
  idx=2 Zn (26.121, 0.000, 8.000)
  idx=30 Zn (0.015, 5.338, 8.003)
  idx=120 Zn (-8.445, 19.940, 8.003)
  idx=131 Zn (8.430, 19.965, 8.003)

Top removed (3):
  idx=423 Zn (4.630, 2.657, 25.573)
  idx=436 Zn (21.505, 2.682, 25.573)
  idx=528 Zn (13.046, 17.283, 25.573)

Total removed: 7 Zn

Inversion partners:
  idx=2 -> UNPAIRED, inv at (-13.060, 22.621, 25.573)
  idx=30 -> UNPAIRED, inv at (13.046, 17.283, 25.570)
  idx=120 -> UNPAIRED, inv at (21.505, 2.682, 25.570)
  idx=131 -> UNPAIRED, inv at (4.630, 2.657, 25.570)
  idx=423 -> UNPAIRED, inv at (8.430, 19.965, 8.000)
  idx=436 -> UNPAIRED, inv at (-8.445, 19.940, 8.000)
  idx=528 -> UNPAIRED, inv at (0.015, 5.338, 8.000)


In [7]:
# From cell 5: 3 bottom Zn are inversion partners of the 3 top Zn (already paired).
# But 1 bottom Zn (idx=2) has no partner — it's an orphan.
# 1 atom can't be split, so we double with a 2x1 supercell.
# This gives 2 orphans, which we can split 1 per surface.

slab_2x = make_supercell(slab, [[2,0,0],[0,1,0],[0,0,1]])

sym2 = np.array(slab_2x.get_chemical_symbols())
mg2, zn2 = sum(sym2 == 'Mg'), sum(sym2 == 'Zn')
z2 = slab_2x.get_positions()[:, 2]

print(f"2x1 supercell:")
print(f"Atoms: {len(slab_2x)}, Mg: {mg2}, Zn: {zn2}")
print(f"Stoichiometric: {'YES' if zn2*21 == mg2*25 else 'NO'}")

# Remove both top and bottom mismatched layers
z_min2 = z2.min()
z_max2 = z2.max()
both_mask2 = (z2 > z_max2 - 0.1) | (z2 < z_min2 + 0.1)
keep2 = np.where(~both_mask2)[0]
trimmed2 = slab_2x[keep2]

ps_trim2 = AseAtomsAdaptor().get_structure(trimmed2)
slab_trim2 = Slab(ps_trim2.lattice, ps_trim2.species, ps_trim2.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps_trim2, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

sym_t2 = np.array(trimmed2.get_chemical_symbols())
print(f"\nAfter removing both surface layers:")
print(f"Atoms: {len(trimmed2)}, Mg: {sum(sym_t2=='Mg')}, Zn: {sum(sym_t2=='Zn')}")
print(f"Symmetric: {slab_trim2.is_symmetric()}")

if slab_trim2.is_symmetric():
    sga2 = SpacegroupAnalyzer(ps_trim2, symprec=0.1)
    print(f"Space group: {sga2.get_space_group_symbol()}")
    for op in sga2.get_symmetry_operations():
        if op.rotation_matrix[2][2] < 0:
            inv_translation = op.translation_vector
            print(f"Inversion: f -> {inv_translation} - f")
            break
    
    removed2 = np.where(both_mask2)[0]
    rem_mg = sum(sym2[j] == 'Mg' for j in removed2)
    rem_zn = sum(sym2[j] == 'Zn' for j in removed2)
    print(f"\nRemoved: {len(removed2)} atoms (Mg{rem_mg} Zn{rem_zn})")

2x1 supercell:
Atoms: 1104, Mg: 504, Zn: 600
Stoichiometric: YES

After removing both surface layers:
Atoms: 1090, Mg: 504, Zn: 586
Symmetric: True
Space group: P-3
Inversion: f -> [0. 0. 0.] - f

Removed: 14 atoms (Mg0 Zn14)


In [8]:
# 14 Zn removed. Inversion is f -> -f (mod 1).
# From the 1x1 analysis: 3 bot pair with 3 top naturally.
# Doubled: 6 bot pair with 6 top. Remaining 2 are orphans to split.

removed_idx = np.where(both_mask2)[0].tolist()
ps_orig2 = AseAtomsAdaptor().get_structure(slab_2x)

print(f"Removed atoms ({len(removed_idx)}):")
bot_rem = [j for j in removed_idx if z2[j] < z_min2 + 0.1]
top_rem = [j for j in removed_idx if z2[j] > z_max2 - 0.1]
print(f"  Bottom: {len(bot_rem)} Zn")
print(f"  Top: {len(top_rem)} Zn")

# Check which are already inversion partners of each other
keep_set2 = set(keep2.tolist())
paired = []
unpaired_bot = []
unpaired_top = []

for j in bot_rem:
    frac = ps_orig2[j].frac_coords
    inv_frac = (-frac) % 1.0
    inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
    
    # Check against top removed atoms
    matched = False
    for k in top_rem:
        d = np.linalg.norm(slab_2x.positions[k] - inv_cart)
        if d < 0.5:
            paired.append((j, k))
            matched = True
            break
    if not matched:
        unpaired_bot.append(j)

used_top = set(k for _, k in paired)
unpaired_top = [j for j in top_rem if j not in used_top]

print(f"\nAlready paired (bot<->top): {len(paired)}")
for b, t in paired:
    print(f"  {b} <-> {t}")
print(f"Unpaired bottom: {len(unpaired_bot)}")
for j in unpaired_bot:
    pos = slab_2x.positions[j]
    print(f"  idx={j} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")
print(f"Unpaired top: {len(unpaired_top)}")
for j in unpaired_top:
    pos = slab_2x.positions[j]
    print(f"  idx={j} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

Removed atoms (14):
  Bottom: 8 Zn
  Top: 6 Zn

Already paired (bot<->top): 6
  30 <-> 1080
  120 <-> 988
  131 <-> 975
  582 <-> 528
  672 <-> 436
  683 <-> 423
Unpaired bottom: 2
  idx=2 (26.121, 0.000, 8.000)
  idx=554 (-0.000, 0.000, 8.000)
Unpaired top: 0


In [9]:
# 6 pairs already exist (stay in slab as-is).
# 2 orphan Zn on bottom: idx=2 and idx=554.
# Move idx=554 to inversion partner of idx=2 (top surface).
# Keep idx=2 on bottom.

sc_fixed = slab_2x.copy()

frac_2 = ps_orig2[2].frac_coords
inv_frac_2 = (-frac_2) % 1.0
inv_cart_2 = ps_orig2.lattice.get_cartesian_coords(inv_frac_2)

old = sc_fixed.positions[554].copy()
sc_fixed.positions[554] = inv_cart_2
print(f"Moved 554 -> inv(2): ({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
      f"({inv_cart_2[0]:.3f}, {inv_cart_2[1]:.3f}, {inv_cart_2[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Moved 554 -> inv(2): (-0.000, 0.000, 8.000) -> (13.060, 22.621, 25.573)

Atoms: 1104, Mg: 504, Zn: 600
Stoichiometric: YES
Symmetric: True


In [10]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [11]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_001_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_001_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg21zn25_001_2layers_reconstructed_ase.data
  Atoms: 1104
  Thickness: 17.6 A
  Area: 1181.8 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [12]:
#unrelaxed surface energy calculation
E_slab =      -1356.3941    # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 1104 / 46                # formula units in slab =
A = 1181.8              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -1453.88360 eV
E_slab - n*E_bulk = 97.48950 eV
S = 0.041246 eV/Ang^2
S = 0.6608 J/m^2


In [13]:
#relaxed surface energy calculation
E_slab_relaxed =  -1366.11623489881  # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 1104 / 46                      # 32 formula units
A = 1181.8                  # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6608
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 87.76737 eV
S relaxed = 0.037133 eV/Ang^2
S relaxed = 0.5949 J/m^2

Unrelaxed S = 0.6608 J/m^2
Relaxed S   = 0.5949 J/m^2
Relaxation energy = 0.0659 J/m^2


In [14]:
# Same workflow: generate, 2x1 supercell, remove both surface layers,
# get inversion, pair atoms, move 1 orphan to top

slab4 = surface(alloy, (0, 0, 1), 4, vacuum=8)
slab4_2x = make_supercell(slab4, [[2,0,0],[0,1,0],[0,0,1]])

z = slab4_2x.get_positions()[:, 2]
sym = np.array(slab4_2x.get_chemical_symbols())
order = np.argsort(z)
print(f"Atoms: {len(slab4_2x)}, Thickness: {z.max()-z.min():.1f} A")

# Cluster layers
z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))
n = len(layers)

# Remove both surface layers and check symmetry
z_min, z_max = z.min(), z.max()
both_mask = (z > z_max - 0.1) | (z < z_min + 0.1)
keep = np.where(~both_mask)[0]
trimmed = slab4_2x[keep]

ps_trim = AseAtomsAdaptor().get_structure(trimmed)
slab_trim = Slab(ps_trim.lattice, ps_trim.species, ps_trim.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps_trim, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"After removing both surface layers: Symmetric = {slab_trim.is_symmetric()}")

if not slab_trim.is_symmetric():
    # Try other removal combos
    print("Trying other removals...")
    for bot_rm in range(0, 6):
        for top_rm in range(0, 6):
            if bot_rm == 0 and top_rm == 0:
                continue
            k = []
            for j in range(bot_rm, n - top_rm):
                k.extend(layer_idx[j])
            if len(k) == 0:
                continue
            tr = slab4_2x[k]
            try:
                ps_tr = AseAtomsAdaptor().get_structure(tr)
                slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                    miller_index=(0,0,1), oriented_unit_cell=ps_tr, shift=0,
                    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                if slab_tr.is_symmetric():
                    removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                    removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                    print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                          f"removed {removed_bot}+{removed_top}")
                    # Use this
                    trimmed = tr
                    ps_trim = ps_tr
                    keep = k
                    both_mask = np.ones(len(slab4_2x), dtype=bool)
                    for idx in k:
                        both_mask[idx] = False
                    break
            except:
                continue
        else:
            continue
        break

# Get inversion
sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")
for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

# Find removed atoms
removed_idx = [i for i in range(len(slab4_2x)) if i not in set(keep)]
print(f"\nRemoved: {len(removed_idx)} atoms")

# Find paired vs unpaired
ps_orig = AseAtomsAdaptor().get_structure(slab4_2x)
keep_set = set(keep) if isinstance(keep, list) else set(keep.tolist())
bot_rem = [j for j in removed_idx if z[j] < z_min + 0.1]
top_rem = [j for j in removed_idx if z[j] > z_max - 0.1]

paired = []
for j in bot_rem:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    for k in top_rem:
        if np.linalg.norm(slab4_2x.positions[k] - inv_cart) < 0.5:
            paired.append((j, k))
            break

used_top = set(k for _, k in paired)
unpaired_bot = [j for j in bot_rem if j not in [b for b, _ in paired]]
unpaired_top = [j for j in top_rem if j not in used_top]

print(f"Paired: {len(paired)}, Unpaired bot: {len(unpaired_bot)}, Unpaired top: {len(unpaired_top)}")

# Reconstruct: move 1 orphan to inv partner of the other
sc_fixed4 = slab4_2x.copy()

if len(unpaired_bot) == 2 and len(unpaired_top) == 0:
    # Same as 2-layer: move one bot orphan to inv(other bot orphan)
    keep_idx = unpaired_bot[0]
    move_idx = unpaired_bot[1]
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed4.positions[move_idx].copy()
    sc_fixed4.positions[move_idx] = inv_cart
    print(f"\nMoved {move_idx} -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed4.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed4 = AseAtomsAdaptor().get_structure(sc_fixed4)
slab_fixed4 = Slab(ps_fixed4.lattice, ps_fixed4.species, ps_fixed4.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps_fixed4, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick4 = sc_fixed4.get_positions()[:,2].max() - sc_fixed4.get_positions()[:,2].min()
area4 = np.linalg.norm(np.cross(sc_fixed4.cell[0], sc_fixed4.cell[1]))

print(f"\nAtoms: {len(sc_fixed4)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*21 == mg_f*25 else 'NO'}")
print(f"Symmetric: {slab_fixed4.is_symmetric()}")
print(f"Thickness: {thick4:.1f} A")
print(f"Area: {area4:.1f} A²")

Atoms: 2208, Thickness: 35.1 A
After removing both surface layers: Symmetric = True
SG = P-3
Inversion: f -> [0. 0. 0.] - f

Removed: 14 atoms
Paired: 6, Unpaired bot: 2, Unpaired top: 0

Moved 1106 -> inv(2): (-0.000, 0.000, 8.000) -> (13.060, 22.621, 43.149)

Atoms: 2208, Mg: 1008, Zn: 1200
Stoichiometric: YES
Symmetric: True
Thickness: 35.1 A
Area: 1181.8 A²


In [15]:
ps = AseAtomsAdaptor().get_structure(sc_fixed4)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg21zn25_001_4layers_reconstructed_ase.data")

print(f"Saved: slabs/mg21zn25_001_4layers_reconstructed_ase.data")

Saved: slabs/mg21zn25_001_4layers_reconstructed_ase.data


In [16]:
#unrelaxed surface energy calculation
E_slab =      -2805.0025    # eV
E_bulk_per_fu = -363.4709  / 6  # eV per formula unit = 
n = 2208 / 46                # formula units in slab =
A = 1181.8              # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -60.578483 eV
n * E_bulk = -2907.76720 eV
E_slab - n*E_bulk = 102.76470 eV
S = 0.043478 eV/Ang^2
S = 0.6966 J/m^2


In [17]:
#relaxed surface energy calculation
E_slab_relaxed =  -2820.04344333143  # eV
E_bulk_per_fu =  -363.4709  / 6  # eV per formula unit
n = 2208 / 46                      # 32 formula units
A = 1181.8                  # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6966
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 87.72376 eV
S relaxed = 0.037114 eV/Ang^2
S relaxed = 0.5946 J/m^2

Unrelaxed S = 0.6966 J/m^2
Relaxed S   = 0.5946 J/m^2
Relaxation energy = 0.1020 J/m^2
